In [3]:
# Get data from: https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page 
from etl_framework.operations.io import LocalFileReader
from etl_framework.operations.pandas.record_extractors.delimited import DelimitedRecordExtractor, NumberField, TimestampField, IntegerField, StringField

In [8]:
record_extractor = DelimitedRecordExtractor(
    fields=[
        IntegerField('vendor_id', column_id='VendorID', size=8),
        TimestampField('pickup_timestamp', column_id='tpep_pickup_datetime'),
        TimestampField('dropoff_timestamp', column_id='tpep_dropoff_datetime'),
        IntegerField('passenger_count', size=8),
        NumberField('distance', column_id='trip_distance'),
        IntegerField('pickup_location_id', column_id='PULocationID'),
        IntegerField('dropoff_location_id', column_id='DOLocationID'),
        IntegerField('payment_type', size=8),
        NumberField('fare_amount'),
        NumberField('tip_amount'),
        NumberField('congestion_surcharge'),
    ],
)
read_extract_records = LocalFileReader() >> record_extractor

In [41]:
input_file = r'C:\Users\d763266\Downloads\yellow_tripdata_2021-01.csv'
df = read_extract_records(input_file)
df

C:\Users\d763266\PycharmProjects\etl_framework\etl_framework\operations\pandas\record_extractors\base.py:61: DtypeWarning: Columns (6) have mixed types.Specify dtype option on import or set low_memory=False.
  dataframe = self.create_dataframe(input_data)


,vendor_id,pickup_timestamp,dropoff_timestamp,passenger_count,distance,pickup_location_id,dropoff_location_id,payment_type,fare_amount,tip_amount,congestion_surcharge,_record_number
0,1,2021-01-01 00:30:10,2021-01-01 00:36:12,1,2.10,142,43,2,8.00,0.00,2.5,1
1,1,2021-01-01 00:51:20,2021-01-01 00:52:19,1,0.20,238,151,2,3.00,0.00,0.0,2
2,1,2021-01-01 00:43:30,2021-01-01 01:11:06,1,14.70,132,165,1,42.00,8.65,0.0,3
3,1,2021-01-01 00:15:48,2021-01-01 00:31:01,0,10.60,138,132,1,29.00,6.05,0.0,4
4,2,2021-01-01 00:31:49,2021-01-01 00:48:21,1,4.94,68,33,1,16.50,4.06,2.5,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1369760,<NA>,2021-01-25 08:32:04,2021-01-25 08:49:32,<NA>,8.80,135,82,<NA>,21.84,0.00,0.0,1369761
1369761,<NA>,2021-01-25 08:34:00,2021-01-25 09:04:00,<NA>,5.86,42,161,<NA>,26.67,0.00,0.0,1369762
1369762,<NA>,2021-01-25 08:37:00,2021-01-25 08:53:00,<NA>,4.45,14,106,<NA>,25.29,0.00,0.0,1369763
1369763,<NA>,2021-01-25 08:28:00,2021-01-25 08:50:00,<NA>,10.04,175,216,<NA>,28.24,0.00,0.0,1369764


In [11]:
df.dtypes

vendor_id                         Int8
pickup_timestamp        datetime64[ns]
dropoff_timestamp       datetime64[ns]
passenger_count                   Int8
distance                       float64
pickup_location_id               Int32
dropoff_location_id              Int32
payment_type                      Int8
fare_amount                    float64
tip_amount                     float64
congestion_surcharge           float64
_record_number                  uint32
dtype: object

In [32]:
from etl_framework.operations.pandas import DropRows, Column, IsNull, ColumnMap, TimedeltaToSeconds, SetColumn, DFWhere
remove_bad_data = DropRows((Column('vendor_id') >> IsNull()) | (Column('passenger_count') == 0))

calculate_duration = SetColumn('trip_duration', (Column('dropoff_timestamp') - Column('pickup_timestamp')) >> TimedeltaToSeconds())

add_card_surcharge = DFWhere(Column('payment_type') == 2, SetColumn('fare_amount', Column('fare_amount')*1.1))

In [36]:
calculate_cost_per_passenger_per_minute = SetColumn('cost_per_passenger_per_minute', (Column('fare_amount') + Column('tip_amount'))/(Column('passenger_count') * Column('trip_duration')/60))

In [37]:
transforms = remove_bad_data >> calculate_duration >> add_card_surcharge >> calculate_cost_per_passenger_per_minute

In [38]:
transforms(df)

,vendor_id,pickup_timestamp,dropoff_timestamp,passenger_count,distance,pickup_location_id,dropoff_location_id,payment_type,fare_amount,tip_amount,congestion_surcharge,_record_number,trip_duration,cost_per_passenger_per_minute
0,1,2021-01-01 00:30:10,2021-01-01 00:36:12,1,2.10,142,43,2,8.80,0.00,2.5,1,362.0,1.458564
1,1,2021-01-01 00:51:20,2021-01-01 00:52:19,1,0.20,238,151,2,3.30,0.00,0.0,2,59.0,3.355932
2,1,2021-01-01 00:43:30,2021-01-01 01:11:06,1,14.70,132,165,1,42.00,8.65,0.0,3,1656.0,1.835145
3,2,2021-01-01 00:31:49,2021-01-01 00:48:21,1,4.94,68,33,1,16.50,4.06,2.5,5,992.0,1.243548
4,1,2021-01-01 00:16:29,2021-01-01 00:24:30,1,1.60,224,68,1,8.00,2.35,2.5,6,481.0,1.29106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1244682,2,2021-01-31 23:58:47,2021-02-01 00:04:40,3,0.81,41,74,2,5.50,0.00,0.0,1271409,353.0,0.311615
1244683,2,2021-01-31 23:07:54,2021-01-31 23:19:42,1,3.81,113,141,2,13.75,0.00,2.5,1271410,708.0,1.165254
1244684,2,2021-01-31 23:30:45,2021-01-31 23:35:13,1,1.32,233,237,2,6.60,0.00,2.5,1271411,268.0,1.477612
1244685,2,2021-01-31 23:09:52,2021-01-31 23:51:56,2,10.56,56,68,1,37.50,0.00,0.0,1271412,2524.0,0.445721


In [42]:
full_pipeline = read_extract_records >> transforms
full_pipeline.profile_snakeviz(input_file)

C:\Users\d763266\PycharmProjects\etl_framework\etl_framework\operations\pandas\record_extractors\base.py:61: DtypeWarning: Columns (6) have mixed types.Specify dtype option on import or set low_memory=False.
  dataframe = self.create_dataframe(input_data)


,vendor_id,pickup_timestamp,dropoff_timestamp,passenger_count,distance,pickup_location_id,dropoff_location_id,payment_type,fare_amount,tip_amount,congestion_surcharge,_record_number,trip_duration,cost_per_passenger_per_minute
0,1,2021-01-01 00:30:10,2021-01-01 00:36:12,1,2.10,142,43,2,8.80,0.00,2.5,1,362.0,1.458564
1,1,2021-01-01 00:51:20,2021-01-01 00:52:19,1,0.20,238,151,2,3.30,0.00,0.0,2,59.0,3.355932
2,1,2021-01-01 00:43:30,2021-01-01 01:11:06,1,14.70,132,165,1,42.00,8.65,0.0,3,1656.0,1.835145
3,2,2021-01-01 00:31:49,2021-01-01 00:48:21,1,4.94,68,33,1,16.50,4.06,2.5,5,992.0,1.243548
4,1,2021-01-01 00:16:29,2021-01-01 00:24:30,1,1.60,224,68,1,8.00,2.35,2.5,6,481.0,1.29106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1244682,2,2021-01-31 23:58:47,2021-02-01 00:04:40,3,0.81,41,74,2,5.50,0.00,0.0,1271409,353.0,0.311615
1244683,2,2021-01-31 23:07:54,2021-01-31 23:19:42,1,3.81,113,141,2,13.75,0.00,2.5,1271410,708.0,1.165254
1244684,2,2021-01-31 23:30:45,2021-01-31 23:35:13,1,1.32,233,237,2,6.60,0.00,2.5,1271411,268.0,1.477612
1244685,2,2021-01-31 23:09:52,2021-01-31 23:51:56,2,10.56,56,68,1,37.50,0.00,0.0,1271412,2524.0,0.445721
